# 📝 Worker 01: Text LLM Fine-Tuner & Production Inference Engine
### Part 1 of the Colab AI Studio 4-Role Full-Auto Architecture

This notebook is **100% Full-Auto Ready**:
1. **Real-Time Progress Sync**: Broadcasts every loading stage (`GPU Allocation` → `Dependencies` → `4-bit Qwen2.5-7B + QLoRA` → `Live Cloudflare Tunnel`) directly to your local **Colab AI Studio Dashboard (`http://localhost:3456`)**.
2. **QLoRA 4-bit Fine-Tuner**: Fine-tunes `Qwen/Qwen2.5-7B-Instruct` on your stories, lorebook, and dialogue on a **Tesla T4 GPU (15GB VRAM)**.
3. **Zero-Touch Auto-Link**: Automatically links the live FastAPI + Cloudflare tunnel URL to your website so you can start chatting and writing immediately.

In [ ]:
# @title Stage 1: GPU Hardware Allocation & Live Progress Telemetry
import os, sys, time, json, subprocess, gc, urllib.request

PAIRING_TOPIC = 'colab_sync_studio_9f4k2x8m'
PROMPT_DRIVE_MOUNT = False # @param {type:"boolean"}

def broadcast_progress(stage_index, stage_name, progress_pct, detail=''):
    print(f"📊 [STAGE {stage_index} | {progress_pct}%] {stage_name} — {detail}")
    payload = json.dumps({
        'type': 'colab_stage',
        'stageIndex': stage_index,
        'stageName': stage_name,
        'progress': progress_pct,
        'detail': detail,
        'timestamp': time.time()
    }).encode('utf-8')
    try:
        req = urllib.request.Request(
            f'https://ntfy.sh/{PAIRING_TOPIC}',
            data=payload,
            headers={'Title': f'Colab Stage {stage_index} ({progress_pct}%)'}
        )
        urllib.request.urlopen(req, timeout=4)
    except Exception:
        pass

def get_gpu_info():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,memory.used,memory.total,utilization.gpu', '--format=csv,noheader,nounits'],
            universal_newlines=True
        ).strip()
        name, mem_used, mem_total, util = [x.strip() for x in out.split(',')]
        return {'gpuName': name, 'vramUsedMB': float(mem_used), 'vramTotalMB': float(mem_total), 'gpuUtilPct': float(util)}
    except Exception:
        return {'gpuName': 'Tesla T4 (Colab)', 'vramUsedMB': 640, 'vramTotalMB': 15360, 'gpuUtilPct': 15}

gpu_info = get_gpu_info()
broadcast_progress(1, 'Allocated Colab GPU Runtime', 25, f"GPU Ready: {gpu_info['gpuName']} ({int(gpu_info['vramTotalMB'])} MB VRAM)")

if os.path.exists('/content/drive/MyDrive'):
    BASE_DIR = '/content/drive/MyDrive/ColabAIStudio'
    print('📁 Using mounted Google Drive at /content/drive/MyDrive/ColabAIStudio')
elif PROMPT_DRIVE_MOUNT:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        BASE_DIR = '/content/drive/MyDrive/ColabAIStudio'
    except Exception:
        BASE_DIR = '/content/ColabAIStudio'
else:
    BASE_DIR = '/content/ColabAIStudio'
    print('⚡ Fast Zero-Popup Mode active (Storage: /content/ColabAIStudio). Toggle PROMPT_DRIVE_MOUNT=True anytime to mount Google Drive.')

CACHE_DIR = os.path.join(BASE_DIR, 'hf_cache')
DATASET_DIR = os.path.join(BASE_DIR, 'datasets')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
for d in [CACHE_DIR, DATASET_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = CACHE_DIR
broadcast_progress(2, 'Installing AI & Tunnel Dependencies', 40, 'Installing transformers, bitsandbytes, peft, datasets, pycloudflared, fastapi...')
!pip install -q transformers accelerate bitsandbytes peft datasets pycloudflared fastapi uvicorn
broadcast_progress(2, 'Dependencies & Workspace Ready', 55, f'Workspace configured at {BASE_DIR}')
print('✅ Stage 1 Complete!')

In [ ]:
# @title Stage 2: Dataset Loader (Curated Story & Chat Dataset OR Custom JSONL)
DATASET_MODE = "Built-in Creative Novel & Chat Dataset" # @param ["Built-in Creative Novel & Chat Dataset", "Custom Google Drive JSONL"]
CUSTOM_JSONL_PATH = "/content/drive/MyDrive/ColabAIStudio/datasets/my_story_data.jsonl" # @param {type:"string"}

import os, json
from datasets import Dataset

dataset_samples = []

if DATASET_MODE == "Custom Google Drive JSONL" and os.path.exists(CUSTOM_JSONL_PATH):
    with open(CUSTOM_JSONL_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    data = json.loads(line)
                    if "messages" in data:
                        dataset_samples.append({"messages": data["messages"]})
                    elif "prompt" in data and "response" in data:
                        dataset_samples.append({
                            "messages": [
                                {"role": "system", "content": data.get("system", "You are a skilled creative novelist and authentic dialogue partner.")},
                                {"role": "user", "content": data["prompt"]},
                                {"role": "assistant", "content": data["response"]}
                            ]
                        })
                except Exception:
                    pass

if not dataset_samples:
    builtin_data = [
        {
            "system": "You are an acclaimed literary novelist. Prioritize sensory grounding, subtext-rich dialogue, psychological depth, and deliberate pacing. Avoid clichés and formulaic corporate prose.",
            "prompt": "Write an opening scene about an antique clockmaker discovering a watch that was wound into tomorrow.",
            "response": "The fog off Glass Harbor never arrived silently; it clicked against the slate roofs like ten thousand tiny escapement wheels falling out of beat.\n\nKaelen Voss held his loupe to the oil lamp, watching the hairspring inside the Magistrate's timepiece. The brass teeth were coated in pale harbor rime—the unmistakable residue of a temporal slip. Someone had wound this watch forty minutes into tomorrow and left the debt unpaid.\n\n\"You told me it only lost seconds,\" Kaelen said without looking up from his workbench. Across the counter, the courier kept her gloved hands buried in her wool coat.\n\n\"Seconds are expensive near the seawall, Master Voss,\" she replied. \"Can you anchor the gear train before the tide turns, or do I take it to the Foundry?\""
        },
        {
            "system": "You are an uncensored, immersive creative companion and storyteller. Embrace bold pacing, emotional depth, and realistic dialogue without artificial preachiness.",
            "prompt": "The courier warns Kaelen that the Foundry wardens are tracking the artifact. Write the next scene.",
            "response": "Kaelen didn't answer immediately. He set the bronze tweezers down on the velvet mat and listened. Beyond the seawall, the heavy bronze bell of the outer harbor tolled twice—the signal that the sea gates were sealing against the midnight fog.\n\n\"The Foundry already knows it's here,\" Kaelen said, his voice flat. He turned the watch case under the lamp until a hairline fracture along the backplate caught the flame. Inside, suspended in mineral oil, a tiny sliver of amber crystal pulsed with a cold, rhythmic thrum.\n\n\"They have wardens at the North Wharf,\" the courier whispered, stepping away from the rain-lashed window. \"If they find us with an unanchored ledger, they won't ask who wound the mainspring. They'll smelt the whole workshop with us inside it.\""
        },
        {
            "system": "You are a natural, intelligent, and engaging conversational companion. Speak like a real person: warm, authentic, articulate, and direct. Avoid repetitive markdown templates.",
            "prompt": "What makes a story protagonist truly memorable rather than generic?",
            "response": "Memorable protagonists aren't defined by their special powers or their tragic backstories—they're defined by their contradictions. A generous character who is secretly petty when embarrassed, or a ruthless survivor who risks everything for something trivial like a stray dog, feels immediately human. Real people don't operate on clean moral spreadsheets; they have hypocrisies, blind spots, and specific habits when they're under stress. Give your character a distinct worldview and let them make decisions that cost them something real."
        },
        {
            "system": "You are an expert prompt engineer. Expand the user's raw prompt into a rich, professional instruction for cinematic generation.",
            "prompt": "Enhance and expand this prompt for IMAGE: 'flying car over neon city'",
            "response": "Retro-futuristic flying sports vehicle banking through towering neon-lit skyscrapers in heavy rain, holographic billboards reflecting off wet carbon-fiber chassis, volumetric amber headlights cutting through coastal mist, anamorphic lens flare, shot on 35mm Arri Alexa, f/1.8 depth of field, photorealistic cyberpunk concept art, 8k resolution"
        }
    ]
    for item in builtin_data:
        dataset_samples.append({
            "messages": [
                {"role": "system", "content": item["system"]},
                {"role": "user", "content": item["prompt"]},
                {"role": "assistant", "content": item["response"]}
            ]
        })

train_dataset = Dataset.from_list(dataset_samples)
sample_save_path = os.path.join(DATASET_DIR, 'sample_dataset_template.jsonl')
with open(sample_save_path, 'w', encoding='utf-8') as f:
    for s in dataset_samples:
        f.write(json.dumps(s) + '\n')
print(f"✅ Stage 2 Complete: Prepared {len(dataset_samples)} training conversations ({sample_save_path})")

In [ ]:
# @title Stage 3: Load Fast 4-bit Liberated Dark-Roast Model & Optional QLoRA Fine-Tuner
BASE_MODEL_NAME = "RichardErkhov/DavidAU_-_L3.2-Rogue-Creative-Instruct-Uncensored-Abliterated-7B-4bits" # @param ["RichardErkhov/DavidAU_-_L3.2-Rogue-Creative-Instruct-Uncensored-Abliterated-7B-4bits", "RichardErkhov/huihui-ai_-_Qwen2.5-7B-Instruct-abliterated-v2-4bits", "RichardErkhov/DavidAU_-_L3-DARKER-PLANET-Broken-Land-12.15B-4bits", "DavidAU/Qwen3-8B-64k-Context-2X-Josiefied-Uncensored", "DavidAU/Qwen3-4B-2507-Thinking-heretic-abliterated-uncensored"]
OUTPUT_LORA_NAME = "dark_roast_creative_writer_lora" # @param {type:"string"}
RUN_FINETUNING_ON_BOOT = False # @param {type:"boolean"}
NUM_EPOCHS = 2 # @param {type:"integer"}
LORA_R = 16 # @param {type:"integer"}
LORA_ALPHA = 32 # @param {type:"integer"}

import torch, gc
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

broadcast_progress(3, f'Downloading & Loading {BASE_MODEL_NAME}', 68, 'Loading pre-quantized 4-bit NF4 weights into Tesla T4 VRAM...')

cfg = AutoConfig.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
is_pre_quantized = hasattr(cfg, 'quantization_config') and bool(cfg.quantization_config)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

if is_pre_quantized:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
else:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

save_dir = os.path.join(MODELS_DIR, OUTPUT_LORA_NAME)
peft_model = model

if os.path.exists(os.path.join(save_dir, 'adapter_config.json')):
    print(f"✅ Found existing fine-tuned LoRA checkpoint at {save_dir}! Loading adapter...")
    broadcast_progress(3, 'Loading Saved Fine-Tuned LoRA Adapter', 85, f'Loaded {OUTPUT_LORA_NAME} into VRAM')
    peft_model = PeftModel.from_pretrained(model, save_dir)
elif RUN_FINETUNING_ON_BOOT:
    try:
        broadcast_progress(3, 'Fine-Tuning QLoRA Adapter on T4 GPU', 78, f'Training {OUTPUT_LORA_NAME} ({NUM_EPOCHS} epochs)...')
        model = prepare_model_for_kbit_training(model)
        peft_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
        )
        peft_model = get_peft_model(model, peft_config)
        peft_model.print_trainable_parameters()

        def tokenize_fn(example):
            text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
            tok = tokenizer(text, truncation=True, max_length=1024, padding="max_length")
            tok["labels"] = tok["input_ids"].copy()
            return tok

        tokenized_ds = train_dataset.map(tokenize_fn, remove_columns=train_dataset.column_names)
        training_args = TrainingArguments(
            output_dir=save_dir,
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=2,
            learning_rate=2e-4,
            fp16=True,
            logging_steps=1,
            save_strategy="no",
            optim="paged_adamw_8bit",
            report_to="none"
        )
        trainer = Trainer(
            model=peft_model,
            train_dataset=tokenized_ds,
            args=training_args,
            data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        )
        train_result = trainer.train()
        peft_model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"🎉 QLoRA Fine-Tuning Complete! Loss: {train_result.training_loss:.4f}")
    except Exception as train_err:
        print(f"⚠️ Fine-tuning skipped ({train_err}); serving base 4-bit {BASE_MODEL_NAME} directly.")
        peft_model = model

broadcast_progress(3, 'Model Loaded & Ready in GPU VRAM', 88, f'{BASE_MODEL_NAME} (4-bit NF4 + LoRA) Ready')
print('✅ Stage 3 Complete!')

In [ ]:
# @title Stage 4: Quick In-Notebook Sanity Check
TEST_PROMPT = "Write a 2-sentence atmospheric description of Glass Harbor at midnight." # @param {type:"string"}

messages = [
    {"role": "system", "content": "You are an acclaimed literary novelist."},
    {"role": "user", "content": TEST_PROMPT}
]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
peft_model.eval()
with torch.no_grad():
    output_tokens = peft_model.generate(**inputs, max_new_tokens=90, temperature=0.75, do_sample=True, pad_token_id=tokenizer.eos_token_id)
output_text = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("✨ Sanity Check Output:\n", output_text.strip())

In [ ]:
# @title Stage 5: Serve Model with FastAPI & Cloudflare Tunnel (Auto-Link to Website)
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn, threading, time, re, urllib.request, json
from pycloudflared import try_cloudflare

app = FastAPI(title='Colab AI Studio — Text LLM Worker')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

ACTIVE_MODEL_LABEL = f"{BASE_MODEL_NAME} + LoRA ({OUTPUT_LORA_NAME})"

@app.get('/', response_class=HTMLResponse)
def index_page():
    return f'''
    <html><head><title>Colab AI Studio — Text Worker</title><style>
    body {{ background:#090d16; color:#f1f5f9; font-family:sans-serif; margin:0; display:flex; align-items:center; justify-content:center; min-height:100vh; }}
    .card {{ background:#0f172a; border:1px solid #1e293b; border-radius:12px; padding:32px; max-width:520px; text-align:center; box-shadow:0 12px 40px rgba(0,0,0,0.5); }}
    h2 {{ color:#f59e0b; margin:0 0 10px; }}
    .badge {{ display:inline-block; background:#10b981; color:#0f172a; font-weight:700; padding:4px 12px; border-radius:12px; font-size:12px; margin-bottom:14px; }}
    p {{ color:#94a3b8; font-size:14px; margin-bottom:20px; line-height:1.5; }}
    .btn {{ display:inline-block; background:#f59e0b; color:#0f172a; font-weight:700; padding:10px 22px; border-radius:6px; text-decoration:none; }}
    </style></head><body>
    <div class="card">
      <h2>⚡ Text LLM Worker is Active!</h2>
      <div class="badge">Model: {ACTIVE_MODEL_LABEL}</div>
      <p>Your Cloudflare tunnel is linked to your local Colab AI Studio Dashboard.</p>
      <a class="btn" href="http://localhost:3456" target="_blank">Open Colab AI Studio Dashboard 🚀</a>
    </div></body></html>
    '''

@app.get('/health')
def health():
    return {
        'status': 'ready',
        'model': {
            'active_model_id': ACTIVE_MODEL_LABEL,
            'modality': 'text',
            'quantization': '4-bit NF4 + LoRA',
            'pipeline_ready': True
        },
        'gpu': get_gpu_info()
    }

@app.post('/generate/chat')
async def generate_chat(req: Request):
    body = await req.json()
    prompt = body.get('prompt', '').strip()
    sys_prompt = body.get('systemPrompt', '').strip() or 'You are a natural, intelligent, and engaging conversational companion. Speak warmly, authentically, and conversationally.'
    temp = float(body.get('temperature', 0.75))
    top_p = float(body.get('topP', 0.9))
    max_tokens = int(body.get('maxTokens', 1024))
    
    msgs = []
    if sys_prompt:
        msgs.append({'role': 'system', 'content': sys_prompt})
    msgs.append({'role': 'user', 'content': prompt})
    
    try:
        prompt_str = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_str, return_tensors='pt').to('cuda')
        peft_model.eval()
        with torch.no_grad():
            output_tokens = peft_model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temp,
                top_p=top_p,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        reply = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return {'ok': True, 'response': reply.strip()}
    except Exception as e:
        return {'ok': False, 'error': str(e), 'response': f'Generation error: {str(e)}'}

@app.post('/shutdown')
def shutdown_runtime():
    try:
        from google.colab import runtime
        runtime.unassign()
        return {'ok': True, 'message': 'Colab runtime unassigned to save Compute Units.'}
    except Exception as e:
        return {'ok': False, 'error': str(e)}

def start_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')

threading.Thread(target=start_server, daemon=True).start()
time.sleep(2)

broadcast_progress(4, 'Starting FastAPI & Cloudflare Reverse Tunnel', 94, 'Binding public HTTPS tunnel to port 8000...')
print('=' * 65)
print('🚀 LAUNCHING FREE CLOUDFLARE REVERSE TUNNEL...')
print('=' * 65)

tunnel_url = None
try:
    tunnel = try_cloudflare(port=8000)
    raw_str = str(tunnel)
    url_match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', raw_str)
    if url_match:
        tunnel_url = url_match.group(0)
    elif hasattr(tunnel, 'tunnel_url'):
        tunnel_url = str(tunnel.tunnel_url)
    elif hasattr(tunnel, 'url'):
        tunnel_url = str(tunnel.url)
        
    print('\n✅ YOUR LIVE COLAB TEXT TUNNEL URL:')
    print(f'👉 {tunnel_url}\n')
    print('=' * 65)
    
    if tunnel_url:
        broadcast_progress(4, 'Colab GPU Tunnel Connected & Live!', 100, tunnel_url)
        try:
            req = urllib.request.Request(
                f'https://ntfy.sh/{PAIRING_TOPIC}',
                data=tunnel_url.encode('utf-8'),
                headers={'Title': 'Colab Text Tunnel Active', 'Tags': 'rocket,gpu'}
            )
            urllib.request.urlopen(req, timeout=5)
            print(f'📡 [Auto-Fetch] Tunnel announced safely to topic "{PAIRING_TOPIC}"!')
            print('👉 Your Colab AI Studio website (http://localhost:3456) is now LIVE and linked!')
        except Exception as dispatch_err:
            print(f'Auto-dispatch notice: {dispatch_err}')
except Exception as e:
    print(f'Tunnel notice: {e}')

print('🟢 GPU WORKER ACTIVE — Keep this cell running while using http://localhost:3456 !', flush=True)
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('⏹️ GPU Worker stopped by user.')